# Online Handoff
Configuration: `stages/01_online_handoff/config`. Figures remain local; datasets are under `experiment/data/`.


### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
online_handoff_experiment_dir = ROOT / 'experiment/research/osd/stages/01_online_handoff'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

online_handoff_data_paths = ExperimentPaths(ROOT, online_handoff_experiment_dir / 'config/paths.json')


# BP dynamics–guided online OSD handoff

- All internal BP trajectories come from this notebook's `trace_min_sum`; wrapped decoders are not used to obtain BP dynamics.
- `standard_BPOSD` stops at the first syndrome-valid hard decision; full OSD runs on the final reliability only if no valid hard decision is found by `T_max`.
- `forced_endpoint_*` always runs OSD on the same reliability snapshot at the standard endpoint; `final_Tmax_*` uses the snapshot at the end of the fixed iteration budget.
- Correctness requires both syndrome validity and a logical sector matching the sampled truth; `exact_error_match` is an additional diagnostic only.
- The primary mechanism study is restricted to d=5. d=9 is disabled by default and may be promoted from exploratory to confirmatory only when the strict-gap count reaches the configured threshold.
- The event-driven mechanism cohort is not used to estimate the overall LER; the final LER comes only from independent unbiased Monte Carlo.


In [ ]:
from dataclasses import dataclass
from hashlib import sha256
from itertools import product
from math import comb
from pathlib import Path
from time import perf_counter
import json
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import kendalltau
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from tqdm.auto import tqdm
from beliefmatching import detector_error_model_to_check_matrices
from codes.ca_code import CA_9_2_5, TriangularCode
from decoders.cpp_backend import Tesseract
from decoders.cython.cython_core.osd_core import ordered_statistics_postprocess
from noises_ca import CircuitBuilder, ErrorRate

online_handoff_config = json.loads((online_handoff_experiment_dir / 'config/bp_dynamics_online_osd_handoff_experiments.json').read_text(encoding='utf-8'))
if online_handoff_config['smoke_mode']:
    online_handoff_config.update(
        mechanism_p_values=[0.05],
        mechanism_gap_target=2,
        mechanism_control_target=2,
        mechanism_batch_size=40,
        mechanism_max_shots_per_point=80,
        final_p_values=[0.05],
        final_min_shots_per_point=20,
        final_max_shots_per_point=20,
        final_batch_size=20,
        final_min_gap_events=1,
        bp_max_iterations=8,
        feature_windows=[3],
        primary_window=3,
        top_k_values=[5],
        primary_top_k=5,
        handoff_thresholds=[0.5, 0.8],
        handoff_persistence=[1, 2],
        fixed_handoff_iterations=[2, 4],
        maximum_wait_iterations=[4, 8],
        prefix_buffer_sizes=[3, -1],
        adaptive_order_thresholds=[0.5, 0.8],
        scaling_values=[1.0],
        damping_values=[0.0],
        parameter_scan_case_limit=4,
        parameter_scan_unbiased_limit=10,
        cv_folds=2,
        bootstrap_repetitions=50,
        temporal_shuffle_repetitions=2,
    )

# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.
# reuse_existing_data: boolean (True or False).
# primary_level: 2.
# mechanism_levels: [2].
# optional_generalization_levels: values restricted to {3}.
# mechanism_p_values, final_p_values: each item is in (0, 1).
# full_osd_method: 'osd_cs'.
# full_osd_combination_sweep_setting: 'all_information_set_subsets_weight_leq_order'.
# schedule: 'parallel'.
# primary_window: chosen from feature_windows.
# primary_top_k: chosen from top_k_values.
# damping_values: each item is in [0, 1).
# scaling_values: each item is positive.
# development_fraction: in (0, 1).
experiment_name = 'osd_handoff_dynamics_smoke' if online_handoff_config['smoke_mode'] else 'osd_handoff_dynamics'
online_handoff_out = online_handoff_data_paths.run(smoke_mode=online_handoff_config['smoke_mode'])

from decoders.cython.cython_core.osd_core import OSD_VERSION, osd_candidate_count
from experiment.utils.misc import check_algorithm_cache
check_algorithm_cache(online_handoff_data_paths, online_handoff_config, {'osd': OSD_VERSION})

from experiment.utils.bp_trace import check_bp_cache
check_bp_cache(online_handoff_data_paths, online_handoff_config)

online_handoff_stats, online_handoff_trace_dir, online_handoff_conf = (online_handoff_data_paths.output('stats'), online_handoff_data_paths.output('trace'), online_handoff_data_paths.output('config'))

for online_handoff_directory in (online_handoff_stats, online_handoff_trace_dir, online_handoff_conf):
    online_handoff_directory.mkdir(parents=True, exist_ok=True)

online_handoff_config_path = online_handoff_data_paths.output('config_json')
if not online_handoff_config_path.is_file() or not online_handoff_config['reuse_existing_data']:
    online_handoff_config_path.write_text(json.dumps({'config': online_handoff_config}, indent=2), encoding='utf-8')

from experiment.utils import misc

def online_handoff_log(level, payload):
    return misc.log(level, payload, CONFIG=online_handoff_config)

def online_handoff_stage_cached(paths):
    return misc.stage_cached(paths, CONFIG=online_handoff_config)

online_handoff_save_csv = misc.save_csv
online_handoff_read_csv = misc.read_csv

def online_handoff_split_for(shot_id):
    raw = int.from_bytes(sha256(f"{online_handoff_config['random_seed']}:{shot_id}".encode()).digest()[:8], 'little') / 2 ** 64
    return 'development' if raw < online_handoff_config['development_fraction'] else 'test'

sns.set_theme(style='whitegrid')
online_handoff_log(2, {'output': str(online_handoff_out)})


## Core decoder instrumentation

`trace_min_sum` exposes edge messages, posterior LLRs, the hard decision and syndrome validity at each iteration. OSD reads only the reliability ordering of the supplied snapshot.

Run the shared definitions before constructing cohorts or snapshots. The instrumentation exposes diagnostic history, while the runtime policies below are restricted to the prefix available at their decision time.


In [ ]:
from experiment.utils import osd_reliability

online_handoff_Context = osd_reliability.Context
online_handoff_BPTrace = osd_reliability.BPTrace
CONTEXT_CACHE = {}
online_handoff_gf2_rank = osd_reliability.gf2_rank

def online_handoff_context_for(level, p):
    key = (int(level), float(p))
    if key in CONTEXT_CACHE:
        return CONTEXT_CACHE[key]
    code_object = CA_9_2_5() if level == 2 else TriangularCode(level)
    circuit = CircuitBuilder(code_object).get_circuit_level_circuit(ErrorRate(p=float(p), p_meas=0), d_rounds=1)
    dem = circuit.detector_error_model()
    matrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
    H = matrices.check_matrix.toarray().astype(np.uint8) % 2
    L = matrices.observables_matrix.toarray().astype(np.uint8) % 2
    priors = np.asarray(matrices.priors, dtype=float)
    checks, variables = np.nonzero(H)
    checks, variables = (checks.astype(np.int32), variables.astype(np.int32))
    edge_ids = np.arange(len(checks), dtype=np.int32)
    check_edges = tuple((edge_ids[checks == check] for check in range(H.shape[0])))
    variable_edges = tuple((edge_ids[variables == variable] for variable in range(H.shape[1])))
    distance = 5 if level == 2 else int(code_object.d)
    assert distance in {5, 9} and online_handoff_gf2_rank(H) == H.shape[0]
    assert H.shape[1] == L.shape[1] == len(priors) and np.all((priors > 0) & (priors < 1))
    q = online_handoff_config['full_osd_order_by_distance'][str(distance)]
    expected = sum((comb(H.shape[1] - H.shape[0], w) for w in range(q + 1)))
    assert expected == online_handoff_config['full_osd_candidate_budget_by_distance'][str(distance)]
    result = online_handoff_Context(level, distance, float(p), H, L, priors, np.log((1 - priors) / priors), dem, checks, variables, check_edges, variable_edges)
    CONTEXT_CACHE[key] = result

    return result

def online_handoff_trace_min_sum(
    context,
    syndrome,
    iterations,
    scaling=1.0,
    damping=0.0,
    schedule='parallel',
):
    assert schedule == 'parallel'
    trace = context.bp_decoder.trace(
        syndrome, iterations, scaling=scaling, damping=damping,
        include_initial_valid=False, combine_scaling_damping=True,
        accumulation='sum_then_prior',
    )
    return online_handoff_BPTrace(trace.c2v, trace.v2c, trace.llr, trace.hard, trace.valid, trace.runtime_seconds, trace.first_valid)

from experiment.utils import misc

online_handoff_bits = misc.bits_variant_2

def online_handoff_bit_array(text, length):
    value = str(text).strip().zfill(length)
    assert len(value) == length and set(value) <= {'0', '1'}
    return np.fromiter((int(x) for x in value), dtype=np.uint8, count=length)

logical_label = osd_reliability.logical_label

def online_handoff_outcome(
    context,
    correction,
    syndrome,
    truth,
    error=None,
):
    correction = np.asarray(correction, dtype=np.uint8)
    valid = np.array_equal(context.H @ correction % 2, syndrome)
    correct = valid and np.array_equal(logical_label(context, correction), truth)
    exact = np.nan if error is None else int(np.array_equal(correction, error))

    return (int(valid), int(correct), exact)

def ordering_proxy(reliability):
    reliability = np.asarray(reliability, dtype=float)
    ordering = np.lexsort((np.arange(len(reliability)), reliability))
    proxy = np.empty(len(reliability), dtype=float)
    proxy[ordering] = np.arange(len(reliability), dtype=float)

    return proxy

def osd_from_reliability(
    context,
    syndrome,
    reliability,
    order,
):
    started = perf_counter()
    correction = ordered_statistics_postprocess(np.ascontiguousarray(reliability, dtype=np.float64), np.ascontiguousarray(syndrome, dtype=np.uint8), np.ascontiguousarray(context.H, dtype=np.uint8), int(order), np.ascontiguousarray(context.priors, dtype=np.float64))
    assert np.array_equal(context.H @ correction % 2, syndrome)

    return (correction, perf_counter() - started)

def online_handoff_full_order(context):
    return osd_reliability.full_order(context, CONFIG=online_handoff_config)

def online_handoff_candidate_count(context, order):
    information_bits = context.H.shape[1] - context.H.shape[0]
    return osd_candidate_count(information_bits, order)

online_handoff_endpoint_of = osd_reliability.endpoint_of

def decode_bundle(
    context,
    syndrome,
    truth,
    error,
    trace,
):
    endpoint = online_handoff_endpoint_of(trace)
    full = online_handoff_full_order(context)
    forced = {}
    final = {}

    for order in sorted({0, 1, full}):
        forced[order] = osd_from_reliability(context, syndrome, trace.llr[endpoint], order)
        final[order] = osd_from_reliability(context, syndrome, trace.llr[-1], order)

    if trace.valid[endpoint]:
        standard_correction, standard_osd_runtime = (trace.hard[endpoint], 0.0)
    else:
        standard_correction, standard_osd_runtime = forced[full]
    standard = online_handoff_outcome(context, standard_correction, syndrome, truth, error)

    return (endpoint, standard, standard_osd_runtime, forced, final)

def online_handoff_tesseract_predictions(context, syndromes):
    return osd_reliability.tesseract_predictions(context, syndromes, CONFIG=online_handoff_config, Tesseract=Tesseract)

def online_handoff_paired_gap_interval(
    standard_correct,
    reference_correct,
    repetitions=None,
):
    standard_error = 1 - np.asarray(standard_correct, dtype=float)
    reference_error = 1 - np.asarray(reference_correct, dtype=float)
    difference = standard_error - reference_error
    rng = np.random.default_rng(online_handoff_config['random_seed'] + 41)
    count = online_handoff_config['bootstrap_repetitions'] if repetitions is None else repetitions
    boot = np.asarray([rng.choice(difference, size=len(difference), replace=True).mean() for _ in range(count)])

    return (float(difference.mean()), float(np.quantile(boot, 0.025)), float(np.quantile(boot, 0.975)))


## Cohort construction

This section generates the event-driven mechanism population and an independent unbiased population. The strict gap `G`, matched success control `A` and residual full-OSD failure control `C` are defined strictly using the standard, forced and Tesseract results.

Keep mechanism-enriched events, matched controls and independent unbiased shots distinct. Achieved event counts determine how much evidence is available for each distance.


In [ ]:
MECHANISM_PATH = online_handoff_data_paths.output('mechanism_population_csv')
COHORT_PATH = online_handoff_data_paths.output('mechanism_cohort_csv')
FINAL_CASE_PATH = online_handoff_data_paths.output('final_unbiased_cases_csv')

def sample_decoded_batch(
    level,
    p,
    shots,
    seed,
    prefix,
):
    context = online_handoff_context_for(level, p)
    sampler = context.dem.compile_sampler(seed=seed)
    syndromes, truths, errors = sampler.sample(shots=shots, return_errors=True)
    tess, tess_runtime = online_handoff_tesseract_predictions(context, syndromes)
    rows = []
    iterator = zip(syndromes, truths, errors, tess)

    for local, (syndrome, truth, error, tess_prediction) in enumerate(tqdm(iterator, total=shots, desc=f'trace d={context.distance} p={p:g}', disable=online_handoff_config['print_level'] == 1)):
        syndrome = np.asarray(syndrome, dtype=np.uint8)
        truth = np.asarray(truth, dtype=np.uint8)
        error = np.asarray(error, dtype=np.uint8)
        trace = online_handoff_trace_min_sum(context, syndrome, online_handoff_config['bp_max_iterations'])
        endpoint, standard, standard_osd_runtime, forced, final = decode_bundle(context, syndrome, truth, error, trace)
        full = online_handoff_full_order(context)
        forced_outcomes = {q: online_handoff_outcome(context, forced[q][0], syndrome, truth, error) for q in forced}
        final_outcomes = {q: online_handoff_outcome(context, final[q][0], syndrome, truth, error) for q in final}
        tess_ok = int(np.array_equal(tess_prediction, truth))

        if not standard[1] and forced_outcomes[full][1] and tess_ok:
            group = 'G'
        elif standard[1] and forced_outcomes[full][1] and tess_ok:
            group = 'A'
        elif not standard[1] and (not forced_outcomes[full][1]) and tess_ok:
            group = 'C'
        else:
            group = 'other'

        rows.append({
            'shot_id': f'{prefix}-{local}',
            'level': level,
            'distance': context.distance,
            'p': p,
            'split': online_handoff_split_for(f'{prefix}-{local}'),
            'syndrome': online_handoff_bits(syndrome),
            'true_logical': online_handoff_bits(truth),
            'error': online_handoff_bits(error),
            'syndrome_weight': int(syndrome.sum()),
            'error_weight': int(error.sum()),
            'first_valid': -1 if trace.first_valid is None else trace.first_valid,
            'endpoint': endpoint,
            'standard_correct': standard[1],
            'standard_exact': standard[2],
            'forced_endpoint_osd0_correct': forced_outcomes[0][1],
            'forced_endpoint_osd1_correct': forced_outcomes[1][1],
            'forced_endpoint_full_correct': forced_outcomes[full][1],
            'final_Tmax_osd0_correct': final_outcomes[0][1],
            'final_Tmax_osd1_correct': final_outcomes[1][1],
            'final_Tmax_full_correct': final_outcomes[full][1],
            'tesseract_correct': tess_ok,
            'cohort': group,
            'bp_endpoint_runtime_seconds': trace.runtime_seconds[endpoint],
            'bp_final_runtime_seconds': trace.runtime_seconds[-1],
            'standard_osd_runtime_seconds': standard_osd_runtime,
            'forced_endpoint_full_runtime_seconds': forced[full][1],
            'final_Tmax_full_runtime_seconds': final[full][1],
            'forced_endpoint_osd0_runtime_seconds': forced[0][1],
            'forced_endpoint_osd1_runtime_seconds': forced[1][1],
            'final_Tmax_osd0_runtime_seconds': final[0][1],
            'final_Tmax_osd1_runtime_seconds': final[1][1],
            'tesseract_runtime_seconds': tess_runtime,
        })

    return pd.DataFrame(rows)

if online_handoff_stage_cached([MECHANISM_PATH, COHORT_PATH]):
    mechanism_population = online_handoff_read_csv(MECHANISM_PATH)
    mechanism_cohort = online_handoff_read_csv(COHORT_PATH)
else:
    batches = []
    for online_handoff_level in online_handoff_config['mechanism_levels']:
        for online_handoff_p_index, online_handoff_p in enumerate(online_handoff_config['mechanism_p_values']):
            online_handoff_seen = 0
            counts = {'G': 0, 'A': 0, 'C': 0}
            while online_handoff_seen < online_handoff_config['mechanism_max_shots_per_point']:
                online_handoff_amount = min(online_handoff_config['mechanism_batch_size'], online_handoff_config['mechanism_max_shots_per_point'] - online_handoff_seen)
                frame = sample_decoded_batch(online_handoff_level, online_handoff_p, online_handoff_amount, online_handoff_config['random_seed'] + online_handoff_level * 100003 + online_handoff_p_index * 1009 + online_handoff_seen, f'mechanism-L{online_handoff_level}-p{online_handoff_p_index}-b{online_handoff_seen}')
                batches.append(frame)
                online_handoff_seen += online_handoff_amount
                local_counts = frame.cohort.value_counts().to_dict()
                counts = {key: counts[key] + int(local_counts.get(key, 0)) for key in counts}
                online_handoff_log(2, {
                    'stage': 'mechanism_screen',
                    'distance': int(frame.distance.iloc[0]),
                    'p': online_handoff_p,
                    'shots': online_handoff_seen,
                    'cohorts': counts,
                })

                if counts['G'] >= online_handoff_config['mechanism_gap_target'] and counts['A'] >= online_handoff_config['mechanism_control_target'] and (counts['C'] >= online_handoff_config['mechanism_control_target']):
                    break

    mechanism_population = pd.concat(batches, ignore_index=True)
    online_handoff_selected = []
    for (online_handoff_distance, online_handoff_p), online_handoff_block in mechanism_population.groupby(['distance', 'p'], sort=True):
        gaps = online_handoff_block[online_handoff_block.cohort == 'G'].head(online_handoff_config['mechanism_gap_target'])
        online_handoff_selected.append(gaps)
        for control in ['A', 'C']:
            pool = online_handoff_block[online_handoff_block.cohort == control].copy()
            if not len(gaps):
                pool = pool.head(online_handoff_config['mechanism_control_target'] if online_handoff_config['smoke_mode'] else 0)
            elif len(pool):
                target_weight = gaps[['syndrome_weight', 'error_weight']].median().to_numpy()
                pool['match_distance'] = np.abs(pool[['syndrome_weight', 'error_weight']].to_numpy() - target_weight).sum(axis=1)
                pool = pool.sort_values('match_distance').head(online_handoff_config['mechanism_control_target']).drop(columns='match_distance')

            online_handoff_selected.append(pool)

    mechanism_cohort = pd.concat(online_handoff_selected, ignore_index=True) if online_handoff_selected else mechanism_population.head(0).copy()
    online_handoff_save_csv(mechanism_population, MECHANISM_PATH)
    online_handoff_save_csv(mechanism_cohort, COHORT_PATH)

def enough_for_final_stop(frame):
    if len(frame) < online_handoff_config['final_min_shots_per_point']:
        return False
    gap_events = int(((frame.standard_correct == 0) & (frame.tesseract_correct == 1)).sum())

    if gap_events < online_handoff_config['final_min_gap_events']:
        return False
    _, lower, _ = online_handoff_paired_gap_interval(frame.standard_correct, frame.tesseract_correct, min(200, online_handoff_config['bootstrap_repetitions']))

    return lower > 0

if online_handoff_stage_cached([FINAL_CASE_PATH]):
    final_cases = online_handoff_read_csv(FINAL_CASE_PATH)
else:
    final_blocks = []
    for online_handoff_level in online_handoff_config['final_levels']:
        for online_handoff_p_index, online_handoff_p in enumerate(online_handoff_config['final_p_values']):
            online_handoff_point = []
            online_handoff_seen = 0
            while online_handoff_seen < online_handoff_config['final_max_shots_per_point']:
                online_handoff_amount = min(online_handoff_config['final_batch_size'], online_handoff_config['final_max_shots_per_point'] - online_handoff_seen)
                online_handoff_point.append(sample_decoded_batch(online_handoff_level, online_handoff_p, online_handoff_amount, online_handoff_config['random_seed'] + 7000001 + online_handoff_level * 100003 + online_handoff_p_index * 1009 + online_handoff_seen, f'unbiased-L{online_handoff_level}-p{online_handoff_p_index}-b{online_handoff_seen}'))
                online_handoff_seen += online_handoff_amount
                current = pd.concat(online_handoff_point, ignore_index=True)

                if enough_for_final_stop(current):
                    break

            final_blocks.append(current)
            online_handoff_log(2, {
                'stage': 'unbiased_sampling',
                'distance': int(current.distance.iloc[0]),
                'p': online_handoff_p,
                'shots': len(current),
                'resolved': enough_for_final_stop(current),
            })

    final_cases = pd.concat(final_blocks, ignore_index=True)
    online_handoff_save_csv(final_cases, FINAL_CASE_PATH)

cohort_counts = mechanism_population.groupby(['distance', 'p', 'cohort']).size().rename('events').reset_index()
display(cohort_counts)


## Experiments 1–3: q_min temporal map, causal features and prefix oracle

Each snapshot runs OSD0, OSD1 and full OSD separately from the same posterior LLRs. Feature functions access only the `0..t` prefix.

Map which tested OSD orders succeed at each snapshot, then compare endpoint, prefix and full-trajectory oracles. The truth-based minimum successful order is an offline label rather than a runtime feature.


In [ ]:
QMIN_PATH = online_handoff_data_paths.output('qmin_snapshots_csv')
FEATURE_PATH = online_handoff_data_paths.output('online_snapshot_features_csv')

def least_reliable(reliability, k):
    k = min(int(k), len(reliability))
    return set(np.argsort(np.abs(reliability), kind='stable')[:k].tolist())

def online_handoff_slope(values):
    values = np.asarray(values, dtype=float)
    if len(values) < 2:
        return 0.0

    return float(np.polyfit(np.arange(len(values)), values, 1)[0])

def prefix_features(
    context,
    trace,
    syndrome,
    t,
):
    assert 1 <= t < len(trace.llr)
    scale = float(np.median(np.abs(context.prior_llr))) + 1e-12
    edge_residual = np.mean(np.abs(np.diff(trace.c2v[:t + 1], axis=0)), axis=1) / scale
    edge_max = np.max(np.abs(np.diff(trace.c2v[:t + 1], axis=0)), axis=1) / scale
    belief_motion = np.mean(np.abs(np.diff(trace.llr[:t + 1], axis=0)), axis=1) / scale
    sign_flip = np.mean(np.sign(trace.c2v[1:t + 1]) != np.sign(trace.c2v[:t]), axis=1)
    conflict = np.mean(trace.c2v[1:t + 1] * trace.v2c[1:t + 1] < 0, axis=1)
    current = np.abs(trace.llr[t]) / scale
    previous = np.abs(trace.llr[t - 1]) / scale
    tau = kendalltau(current, previous).statistic
    row = {
        'iteration': t,
        'bp_syndrome_consistent': int(trace.valid[t]),
        'syndrome_residual_weight': int(np.sum(context.H @ trace.hard[t] % 2 ^ syndrome)),
        'time_since_first_valid': -1 if trace.first_valid is None or t < trace.first_valid else t - trace.first_valid,
        'normalized_residual_current': float(edge_residual[-1]),
        'normalized_residual_max_current': float(edge_max[-1]),
        'normalized_belief_motion_current': float(belief_motion[-1]),
        'sign_flip_current': float(sign_flip[-1]),
        'extrinsic_conflict_current': float(conflict[-1]),
        'global_kendall': 0.0 if np.isnan(tau) else float(tau),
        'llr_min': float(current.min()),
        'llr_q10': float(np.quantile(current, 0.1)),
        'llr_q25': float(np.quantile(current, 0.25)),
        'llr_median': float(np.median(current)),
        'llr_max': float(current.max()),
        'llr_spread': float(current.max() - current.min()),
    }

    ranks_now = np.empty(len(current), dtype=int)
    ranks_now[np.argsort(current, kind='stable')] = np.arange(len(current))
    ranks_prev = np.empty(len(previous), dtype=int)
    ranks_prev[np.argsort(previous, kind='stable')] = np.arange(len(previous))

    for k_value in online_handoff_config['top_k_values']:
        k = min(k_value, len(current))
        now = least_reliable(current, k)
        before = least_reliable(previous, k)
        row[f'top{k_value}_jaccard'] = len(now & before) / max(1, len(now | before))
        boundary = np.flatnonzero((ranks_now >= max(0, k - 2)) & (ranks_now < min(len(current), k + 2)))
        row[f'top{k_value}_boundary_motion'] = float(np.mean(np.abs(ranks_now[boundary] - ranks_prev[boundary]))) if len(boundary) else 0.0
        window = min(online_handoff_config['primary_window'], t + 1)
        history = [least_reliable(trace.llr[u], k) for u in range(t - window + 1, t + 1)]
        memberships = [len(now & past) / k for past in history]
        row[f'top{k_value}_persistence_mean'] = float(np.mean(memberships))
        row[f'top{k_value}_persistence_min'] = float(np.min(memberships))

    for window in online_handoff_config['feature_windows']:
        width = min(window, len(edge_residual))
        row[f'residual_mean_w{window}'] = float(np.mean(edge_residual[-width:]))
        row[f'residual_slope_w{window}'] = online_handoff_slope(edge_residual[-width:])
        row[f'belief_motion_mean_w{window}'] = float(np.mean(belief_motion[-width:]))
        row[f'sign_flip_mean_w{window}'] = float(np.mean(sign_flip[-width:]))
        row[f'conflict_mean_w{window}'] = float(np.mean(conflict[-width:]))

    return row

if online_handoff_stage_cached([QMIN_PATH, FEATURE_PATH]):
    qmin_snapshots = online_handoff_read_csv(QMIN_PATH)
    online_snapshot_features = online_handoff_read_csv(FEATURE_PATH)
else:
    q_rows, feature_rows = ([], [])
    for online_handoff_case in tqdm(mechanism_cohort.itertuples(index=False), total=len(mechanism_cohort), desc='snapshot trajectories', disable=online_handoff_config['print_level'] == 1):
        online_handoff_context = online_handoff_context_for(int(online_handoff_case.level), float(online_handoff_case.p))
        online_handoff_syndrome = online_handoff_bit_array(online_handoff_case.syndrome, online_handoff_context.H.shape[0])
        online_handoff_truth = online_handoff_bit_array(online_handoff_case.true_logical, online_handoff_context.L.shape[0])
        online_handoff_error = online_handoff_bit_array(online_handoff_case.error, online_handoff_context.H.shape[1])
        online_handoff_trace = online_handoff_trace_min_sum(online_handoff_context, online_handoff_syndrome, online_handoff_config['bp_max_iterations'])
        online_handoff_endpoint = online_handoff_endpoint_of(online_handoff_trace)
        full = online_handoff_full_order(online_handoff_context)

        for online_handoff_t in range(1, len(online_handoff_trace.llr)):
            online_handoff_outcomes = {}
            runtimes = {}
            for online_handoff_q in sorted({0, 1, full}):
                online_handoff_correction, online_handoff_runtime = osd_from_reliability(online_handoff_context, online_handoff_syndrome, online_handoff_trace.llr[online_handoff_t], online_handoff_q)
                online_handoff_outcomes[online_handoff_q] = online_handoff_outcome(online_handoff_context, online_handoff_correction, online_handoff_syndrome, online_handoff_truth, online_handoff_error)
                runtimes[online_handoff_q] = online_handoff_runtime

            q_min = '0' if online_handoff_outcomes[0][1] else '1' if online_handoff_outcomes[1][1] else 'full' if online_handoff_outcomes[full][1] else 'inf'
            common = {
                'shot_id': online_handoff_case.shot_id,
                'distance': online_handoff_context.distance,
                'p': online_handoff_case.p,
                'split': online_handoff_case.split,
                'cohort': online_handoff_case.cohort,
                'iteration': online_handoff_t,
                'endpoint': online_handoff_endpoint,
                'first_valid': online_handoff_trace.first_valid if online_handoff_trace.first_valid is not None else -1,
            }

            q_rows.append({
                **common,
                'relative_to_first_valid': np.nan if online_handoff_trace.first_valid is None else online_handoff_t - online_handoff_trace.first_valid,
                'bp_syndrome_consistent': int(online_handoff_trace.valid[online_handoff_t]),
                'bp_logical_correct_offline': online_handoff_outcome(online_handoff_context, online_handoff_trace.hard[online_handoff_t], online_handoff_syndrome, online_handoff_truth, online_handoff_error)[1],
                'osd0_correct': online_handoff_outcomes[0][1],
                'osd1_correct': online_handoff_outcomes[1][1],
                'osdfull_correct': online_handoff_outcomes[full][1],
                'osd0_exact': online_handoff_outcomes[0][2],
                'osd1_exact': online_handoff_outcomes[1][2],
                'osdfull_exact': online_handoff_outcomes[full][2],
                'q_min': q_min,
                'osd0_runtime_seconds': runtimes[0],
                'osd1_runtime_seconds': runtimes[1],
                'osdfull_runtime_seconds': runtimes[full],
            })
            feature_rows.append({
                **common,
                **prefix_features(online_handoff_context, online_handoff_trace, online_handoff_syndrome, online_handoff_t),
                'y_osd0': online_handoff_outcomes[0][1],
                'y_osd1': online_handoff_outcomes[1][1],
                'y_osdfull': online_handoff_outcomes[full][1],
            })

    qmin_snapshots = pd.DataFrame(q_rows)
    online_snapshot_features = pd.DataFrame(feature_rows)
    online_handoff_save_csv(qmin_snapshots, QMIN_PATH)
    online_handoff_save_csv(online_snapshot_features, FEATURE_PATH)

def longest_run(values):
    best = current = 0
    for value in values:
        current = current + 1 if value else 0
        best = max(best, current)

    return best

temporal_rows, oracle_rows = ([], [])
for online_handoff_shot_id, online_handoff_block in qmin_snapshots.groupby('shot_id', sort=False):
    online_handoff_block = online_handoff_block.sort_values('iteration')
    online_handoff_endpoint = int(online_handoff_block.endpoint.iloc[0])
    prefix = online_handoff_block[online_handoff_block.iteration <= online_handoff_endpoint]
    online_handoff_row = online_handoff_block.iloc[0]

    for online_handoff_order_name, column in [('osd0', 'osd0_correct'), ('osd1', 'osd1_correct'), ('osdfull', 'osdfull_correct')]:
        online_handoff_values = online_handoff_block[column].astype(int).to_numpy()
        prefix_values = prefix[column].astype(int).to_numpy()
        source_column = {
            'osd0': 'forced_endpoint_osd0_correct',
            'osd1': 'forced_endpoint_osd1_correct',
            'osdfull': 'forced_endpoint_full_correct',
        }[online_handoff_order_name]
        online_handoff_endpoint_success = int(mechanism_cohort.loc[mechanism_cohort.shot_id == online_handoff_shot_id, source_column].iloc[0])
        useful = online_handoff_block.loc[online_handoff_block[column] == 1, 'iteration']
        temporal_rows.append({
            'shot_id': online_handoff_shot_id,
            'distance': online_handoff_row.distance,
            'p': online_handoff_row.p,
            'cohort': online_handoff_row.cohort,
            'osd_order': online_handoff_order_name,
            'first_useful': int(useful.min()) if len(useful) else -1,
            'longest_persistence': longest_run(online_handoff_values),
            'reentry_count': int(np.sum((online_handoff_values[1:] == 1) & (online_handoff_values[:-1] == 0))),
        })
        oracle_rows.append({
            'shot_id': online_handoff_shot_id,
            'distance': online_handoff_row.distance,
            'p': online_handoff_row.p,
            'cohort': online_handoff_row.cohort,
            'osd_order': online_handoff_order_name,
            'endpoint_success': online_handoff_endpoint_success,
            'prefix_oracle_success': int(max(online_handoff_endpoint_success, prefix_values.max() if len(prefix_values) else 0)),
            'full_oracle_success': int(max(online_handoff_endpoint_success, online_handoff_values.max())),
        })

qmin_temporal_summary = pd.DataFrame(temporal_rows).groupby(['distance', 'p', 'cohort', 'osd_order'], as_index=False).agg(
    shots=('shot_id', 'nunique'),
    first_useful_rate=('first_useful', lambda x: np.mean(x >= 0)),
    mean_first_useful=('first_useful', lambda x: np.mean(x[x >= 0]) if np.any(x >= 0) else np.nan),
    mean_longest_persistence=('longest_persistence', 'mean'),
    mean_reentry_count=('reentry_count', 'mean'),
)
prefix_oracle_ceiling = pd.DataFrame(oracle_rows).groupby(['distance', 'p', 'cohort', 'osd_order'], as_index=False).agg(
    shots=('shot_id', 'nunique'),
    endpoint_rate=('endpoint_success', 'mean'),
    prefix_oracle_rate=('prefix_oracle_success', 'mean'),
    full_oracle_rate=('full_oracle_success', 'mean'),
)
online_handoff_save_csv(qmin_temporal_summary, online_handoff_data_paths.output('qmin_temporal_summary_csv'))
online_handoff_save_csv(prefix_oracle_ceiling, online_handoff_data_paths.output('prefix_oracle_ceiling_csv'))

display(prefix_oracle_ceiling)


## Experiments 2, 4–7: held-out models, temporal shuffle, online policy, prefix buffer and adaptive order

All splits and cross-validation use `shot_id` as the grouping variable. Runtime features exclude truth, correctness, q_min and Tesseract outputs.

Fit and evaluate the causal policies, including the time-shuffle control and prefix buffer. The comparison tests whether ordered trajectory information helps beyond individual snapshot quality.


In [ ]:
online_handoff_w = online_handoff_config['primary_window']
K = online_handoff_config['primary_top_k']
STATIC_FEATURES = ['iteration', 'bp_syndrome_consistent', 'syndrome_residual_weight', 'time_since_first_valid', 'llr_min', 'llr_q10', 'llr_q25', 'llr_median', 'llr_max', 'llr_spread']
RANK_FEATURES = ['global_kendall', f'top{K}_jaccard', f'top{K}_boundary_motion', f'top{K}_persistence_mean', f'top{K}_persistence_min']
MESSAGE_FEATURES = ['normalized_residual_current', 'normalized_residual_max_current', 'normalized_belief_motion_current', 'sign_flip_current', 'extrinsic_conflict_current', f'residual_mean_w{online_handoff_w}', f'residual_slope_w{online_handoff_w}', f'belief_motion_mean_w{online_handoff_w}', f'sign_flip_mean_w{online_handoff_w}', f'conflict_mean_w{online_handoff_w}']
online_handoff_feature_sets = {
    'static': STATIC_FEATURES,
    'rank': STATIC_FEATURES + RANK_FEATURES,
    'message': STATIC_FEATURES + MESSAGE_FEATURES,
    'full': STATIC_FEATURES + RANK_FEATURES + MESSAGE_FEATURES,
}

assert not {'y_osd0', 'y_osd1', 'y_osdfull', 'q_min', 'tesseract_correct', 'true_logical'} & set(sum(online_handoff_feature_sets.values(), []))

def make_model(kind):
    if kind == 'logistic':
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced', random_state=online_handoff_config['random_seed']))
    assert kind == 'tree'

    return DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=10,
        class_weight='balanced',
        random_state=online_handoff_config['random_seed'],
    )

def safe_scores(y, score):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)
    if len(np.unique(y)) < 2:
        return (np.nan, np.nan)

    return (float(roc_auc_score(y, score)), float(average_precision_score(y, score)))

def fit_group_model(
    frame,
    features,
    target,
    kind,
):
    development = frame[frame.split == 'development'].copy()
    test = frame[frame.split == 'test'].copy()
    if development[target].nunique() < 2 or test[target].nunique() < 2 or development.shot_id.nunique() < 2:
        return (None, {
            'status': 'insufficient_events',
            'cv_auroc': np.nan,
            'cv_auprc': np.nan,
            'test_auroc': np.nan,
            'test_auprc': np.nan,
        })

    folds = min(online_handoff_config['cv_folds'], development.shot_id.nunique())
    cv = GroupKFold(n_splits=folds)
    oof = np.full(len(development), np.nan)
    X = development[features].fillna(0).to_numpy()
    y = development[target].to_numpy()

    for train, valid in cv.split(X, y, groups=development.shot_id):
        if len(np.unique(y[train])) < 2:
            oof[valid] = float(np.mean(y[train]))
        else:
            model = make_model(kind)
            model.fit(X[train], y[train])
            oof[valid] = model.predict_proba(X[valid])[:, 1]

    cv_roc, cv_pr = safe_scores(y, oof)
    model = make_model(kind)
    model.fit(X, y)
    test_score = model.predict_proba(test[features].fillna(0).to_numpy())[:, 1]
    test_roc, test_pr = safe_scores(test[target], test_score)

    return (model, {
        'status': 'ok',
        'cv_auroc': cv_roc,
        'cv_auprc': cv_pr,
        'test_auroc': test_roc,
        'test_auprc': test_pr,
    })

ablation_rows = []
MODELS = {}
for online_handoff_target in ['y_osd0', 'y_osd1']:
    for online_handoff_feature_set, features in online_handoff_feature_sets.items():
        for online_handoff_kind in ['logistic', 'tree']:
            online_handoff_model, online_handoff_metrics = fit_group_model(online_snapshot_features, features, online_handoff_target, online_handoff_kind)
            MODELS[online_handoff_target, online_handoff_feature_set, online_handoff_kind] = online_handoff_model
            ablation_rows.append({
                'analysis': 'feature_set',
                'target': online_handoff_target,
                'feature_set': online_handoff_feature_set,
                'model': online_handoff_kind,
                'feature': '',
                **online_handoff_metrics,
            })

test_features = online_snapshot_features[online_snapshot_features.split == 'test']
online_handoff_rng = np.random.default_rng(online_handoff_config['random_seed'] + 83)
for online_handoff_target in ['y_osd0', 'y_osd1']:
    for feature in online_handoff_feature_sets['full']:
        y = test_features[online_handoff_target].to_numpy()
        score = test_features[feature].fillna(0).to_numpy()
        if len(np.unique(y)) < 2:
            roc = pr = effect = lo = hi = np.nan
        else:
            roc_forward, pr_forward = safe_scores(y, score)
            roc_reverse, pr_reverse = safe_scores(y, -score)
            roc, pr = (roc_forward, pr_forward) if roc_forward >= roc_reverse else (roc_reverse, pr_reverse)
            effect = float(score[y == 1].mean() - score[y == 0].mean())
            shot_ids = test_features.shot_id.unique()
            boot = []

            for online_handoff_ in range(online_handoff_config['bootstrap_repetitions']):
                online_handoff_selected = online_handoff_rng.choice(shot_ids, len(shot_ids), replace=True)
                sample = pd.concat([test_features[test_features.shot_id == s] for s in online_handoff_selected], ignore_index=True)
                if sample[online_handoff_target].nunique() == 2:
                    vals = sample[feature].fillna(0).to_numpy()
                    labels = sample[online_handoff_target].to_numpy()
                    boot.append(vals[labels == 1].mean() - vals[labels == 0].mean())

            lo, hi = np.quantile(boot, [0.025, 0.975]) if boot else [np.nan, np.nan]

        ablation_rows.append({
            'analysis': 'univariate',
            'target': online_handoff_target,
            'feature_set': 'single',
            'model': 'score',
            'feature': feature,
            'status': 'ok' if np.isfinite(roc) else 'insufficient_events',
            'cv_auroc': np.nan,
            'cv_auprc': np.nan,
            'test_auroc': roc,
            'test_auprc': pr,
            'effect_size': effect,
            'effect_ci_low': lo,
            'effect_ci_high': hi,
        })

snapshot_feature_ablation = pd.DataFrame(ablation_rows)
online_handoff_save_csv(snapshot_feature_ablation, online_handoff_data_paths.output('snapshot_feature_ablation_csv'))
online_handoff_shuffle_rows = []

for online_handoff_repetition in tqdm(range(online_handoff_config['temporal_shuffle_repetitions']), desc='temporal shuffle', disable=online_handoff_config['print_level'] == 1):
    online_handoff_shuffled = online_snapshot_features.copy()
    for online_handoff_, online_handoff_indices in online_handoff_shuffled.groupby('shot_id').groups.items():
        permutation = online_handoff_rng.permutation(online_handoff_indices)
        online_handoff_shuffled.loc[online_handoff_indices, MESSAGE_FEATURES + RANK_FEATURES] = online_handoff_shuffled.loc[permutation, MESSAGE_FEATURES + RANK_FEATURES].to_numpy()

    online_handoff_, online_handoff_metrics = fit_group_model(online_handoff_shuffled, online_handoff_feature_sets['full'], 'y_osd1', 'logistic')
    online_handoff_shuffle_rows.append({'repetition': online_handoff_repetition, **online_handoff_metrics})

temporal_shuffle_control = pd.DataFrame(online_handoff_shuffle_rows)
online_handoff_save_csv(temporal_shuffle_control, online_handoff_data_paths.output('temporal_shuffle_control_csv'))

def score_block(
    block,
    target='y_osd1',
    feature_set='full',
    kind='logistic',
):
    model = MODELS.get((target, feature_set, kind))
    return np.full(len(block), 0.5) if model is None else model.predict_proba(block[online_handoff_feature_sets[feature_set]].fillna(0).to_numpy())[:, 1]

def first_crossing(
    block,
    scores,
    threshold,
    persistence,
    maximum_wait,
    require_valid=False,
):
    consecutive = 0
    for position, row in enumerate(block.itertuples(index=False)):
        if row.iteration > maximum_wait:
            break
        eligible = scores[position] >= threshold and (not require_valid or bool(row.bp_syndrome_consistent))
        consecutive = consecutive + 1 if eligible else 0

        if consecutive >= persistence:
            return (int(row.iteration), 'trigger')

    return (int(block.endpoint.iloc[0]), 'fallback_endpoint')

def first_true_run(flags, persistence):
    consecutive = 0
    for position, flag in enumerate(flags):
        consecutive = consecutive + 1 if flag else 0
        if consecutive >= persistence:
            return position

    return None

def evaluate_snapshot(
    case_row,
    iteration,
    order,
    trace=None,
):
    context = online_handoff_context_for(int(case_row.level), float(case_row.p))
    syndrome = online_handoff_bit_array(case_row.syndrome, context.H.shape[0])
    truth = online_handoff_bit_array(case_row.true_logical, context.L.shape[0])

    if trace is None:
        trace = online_handoff_trace_min_sum(context, syndrome, online_handoff_config['bp_max_iterations'])
    correction, osd_runtime = osd_from_reliability(context, syndrome, trace.llr[int(iteration)], order)

    return (online_handoff_outcome(context, correction, syndrome, truth)[1], osd_runtime, trace)

ONLINE_TRIAL_PATH = online_handoff_trace_dir / 'online_handoff_trials.csv'
online_cache = online_handoff_stage_cached([ONLINE_TRIAL_PATH])
case_lookup = mechanism_cohort.set_index('shot_id')
online_trial_rows = []
online_groups = [] if online_cache else online_snapshot_features.groupby('shot_id', sort=False)

for online_handoff_shot_id, online_handoff_block in tqdm(online_groups, total=0 if online_cache else online_snapshot_features.shot_id.nunique(), desc='online policies', disable=online_handoff_config['print_level'] == 1):
    online_handoff_block = online_handoff_block.sort_values('iteration').reset_index(drop=True)
    online_handoff_case = case_lookup.loc[str(online_handoff_shot_id)]
    online_handoff_scores = score_block(online_handoff_block)
    online_handoff_context = online_handoff_context_for(int(online_handoff_case.level), float(online_handoff_case.p))
    online_handoff_syndrome = online_handoff_bit_array(online_handoff_case.syndrome, online_handoff_context.H.shape[0])
    online_handoff_trace = online_handoff_trace_min_sum(online_handoff_context, online_handoff_syndrome, online_handoff_config['bp_max_iterations'])
    online_handoff_standard_order = -1 if int(online_handoff_case.first_valid) >= 1 else online_handoff_full_order(online_handoff_context)
    standard_runtime = float(online_handoff_case.bp_endpoint_runtime_seconds + online_handoff_case.standard_osd_runtime_seconds)
    online_trial_rows.append({
        'shot_id': online_handoff_shot_id,
        'distance': online_handoff_case.distance,
        'p': online_handoff_case.p,
        'split': online_handoff_case.split,
        'policy_family': 'standard_BPOSD_endpoint',
        'threshold': np.nan,
        'persistence': 1,
        'maximum_wait': int(online_handoff_case.endpoint),
        'require_syndrome_consistent': 1,
        'selected_iteration': int(online_handoff_case.endpoint),
        'selection_reason': 'standard',
        'osd_order': online_handoff_standard_order,
        'logical_correct': int(online_handoff_case.standard_correct),
        'bp_runtime_seconds': float(online_handoff_case.bp_endpoint_runtime_seconds),
        'osd_runtime_seconds': float(online_handoff_case.standard_osd_runtime_seconds),
        'total_runtime_seconds': standard_runtime,
        'candidate_count': 0 if online_handoff_standard_order < 0 else online_handoff_candidate_count(online_handoff_context, online_handoff_standard_order),
    })

    for online_handoff_threshold, persistence, wait, require_valid in product(online_handoff_config['handoff_thresholds'], online_handoff_config['handoff_persistence'], online_handoff_config['maximum_wait_iterations'], [False, True]):
        online_handoff_selected, reason = first_crossing(online_handoff_block, online_handoff_scores, online_handoff_threshold, persistence, wait, require_valid)
        online_handoff_order = 1 if reason == 'trigger' else online_handoff_full_order(online_handoff_context)
        online_handoff_correct, online_handoff_osd_runtime, online_handoff_ = evaluate_snapshot(online_handoff_case, online_handoff_selected, online_handoff_order, online_handoff_trace)
        online_trial_rows.append({
            'shot_id': online_handoff_shot_id,
            'distance': online_handoff_case.distance,
            'p': online_handoff_case.p,
            'split': online_handoff_case.split,
            'policy_family': 'learned_full_logistic',
            'threshold': online_handoff_threshold,
            'persistence': persistence,
            'maximum_wait': wait,
            'require_syndrome_consistent': int(require_valid),
            'selected_iteration': online_handoff_selected,
            'selection_reason': reason,
            'osd_order': online_handoff_order,
            'logical_correct': online_handoff_correct,
            'bp_runtime_seconds': online_handoff_trace.runtime_seconds[online_handoff_selected],
            'osd_runtime_seconds': online_handoff_osd_runtime,
            'total_runtime_seconds': online_handoff_trace.runtime_seconds[online_handoff_selected] + online_handoff_osd_runtime,
            'candidate_count': online_handoff_candidate_count(online_handoff_context, online_handoff_order),
        })

    for online_handoff_fixed in online_handoff_config['fixed_handoff_iterations']:
        online_handoff_selected = min(online_handoff_fixed, len(online_handoff_trace.llr) - 1)
        online_handoff_correct, online_handoff_osd_runtime, online_handoff_ = evaluate_snapshot(online_handoff_case, online_handoff_selected, 1, online_handoff_trace)
        online_trial_rows.append({
            'shot_id': online_handoff_shot_id,
            'distance': online_handoff_case.distance,
            'p': online_handoff_case.p,
            'split': online_handoff_case.split,
            'policy_family': 'fixed_iteration',
            'threshold': online_handoff_fixed,
            'persistence': 1,
            'maximum_wait': online_handoff_fixed,
            'require_syndrome_consistent': 0,
            'selected_iteration': online_handoff_selected,
            'selection_reason': 'fixed',
            'osd_order': 1,
            'logical_correct': online_handoff_correct,
            'bp_runtime_seconds': online_handoff_trace.runtime_seconds[online_handoff_selected],
            'osd_runtime_seconds': online_handoff_osd_runtime,
            'total_runtime_seconds': online_handoff_trace.runtime_seconds[online_handoff_selected] + online_handoff_osd_runtime,
            'candidate_count': online_handoff_candidate_count(online_handoff_context, 1),
        })

    development = online_snapshot_features[online_snapshot_features.split == 'development']
    residual_cut = development[f'residual_mean_w{online_handoff_w}'].quantile(0.5) if len(development) else 0
    stability_cut = development[f'top{K}_jaccard'].quantile(0.5) if len(development) else 1
    handcrafted = {
        'residual_threshold': online_handoff_block[f'residual_mean_w{online_handoff_w}'].to_numpy() <= residual_cut,
        'topk_stability': online_handoff_block[f'top{K}_jaccard'].to_numpy() >= stability_cut,
        'residual_plus_stability': (online_handoff_block[f'residual_mean_w{online_handoff_w}'].to_numpy() <= residual_cut) & (online_handoff_block[f'top{K}_jaccard'].to_numpy() >= stability_cut),
        'conflict_plus_stability': (online_handoff_block[f'conflict_mean_w{online_handoff_w}'].to_numpy() <= development[f'conflict_mean_w{online_handoff_w}'].quantile(0.5)) & (online_handoff_block[f'top{K}_jaccard'].to_numpy() >= stability_cut),
    }

    for family, flags in handcrafted.items():
        persistence_values = [1] if family == 'residual_threshold' else online_handoff_config['handoff_persistence']
        for persistence in persistence_values:
            position = first_true_run(flags, persistence)
            reason = 'rule_trigger' if position is not None else 'fallback_endpoint'
            online_handoff_selected = int(online_handoff_block.iloc[position].iteration) if position is not None else int(online_handoff_case.endpoint)
            online_handoff_order = 1 if position is not None else online_handoff_full_order(online_handoff_context)
            online_handoff_correct, online_handoff_osd_runtime, online_handoff_ = evaluate_snapshot(online_handoff_case, online_handoff_selected, online_handoff_order, online_handoff_trace)
            online_trial_rows.append({
                'shot_id': online_handoff_shot_id,
                'distance': online_handoff_case.distance,
                'p': online_handoff_case.p,
                'split': online_handoff_case.split,
                'policy_family': family,
                'threshold': np.nan,
                'persistence': persistence,
                'maximum_wait': online_handoff_config['bp_max_iterations'],
                'require_syndrome_consistent': 0,
                'selected_iteration': online_handoff_selected,
                'selection_reason': reason,
                'osd_order': online_handoff_order,
                'logical_correct': online_handoff_correct,
                'bp_runtime_seconds': online_handoff_trace.runtime_seconds[online_handoff_selected],
                'osd_runtime_seconds': online_handoff_osd_runtime,
                'total_runtime_seconds': online_handoff_trace.runtime_seconds[online_handoff_selected] + online_handoff_osd_runtime,
                'candidate_count': online_handoff_candidate_count(online_handoff_context, online_handoff_order),
            })

if online_cache:
    online_handoff_trials = online_handoff_read_csv(ONLINE_TRIAL_PATH)
else:
    online_handoff_trials = pd.DataFrame(online_trial_rows)
    online_handoff_save_csv(online_handoff_trials, ONLINE_TRIAL_PATH)

online_handoff_rules = online_handoff_trials.groupby(['distance', 'p', 'split', 'policy_family', 'threshold', 'persistence', 'maximum_wait', 'require_syndrome_consistent'], dropna=False, as_index=False).agg(
    shots=('shot_id', 'nunique'),
    success_rate=('logical_correct', 'mean'),
    mean_selected_iteration=('selected_iteration', 'mean'),
    mean_candidate_count=('candidate_count', 'mean'),
    mean_runtime_seconds=('total_runtime_seconds', 'mean'),
)
online_handoff_save_csv(online_handoff_rules, online_handoff_data_paths.output('online_handoff_rules_csv'))
development_rules = online_handoff_rules[(online_handoff_rules.split == 'development') & (online_handoff_rules.policy_family == 'learned_full_logistic') & (online_handoff_rules.require_syndrome_consistent == 0)]

if len(development_rules):
    production = development_rules.sort_values(['success_rate', 'mean_candidate_count', 'mean_selected_iteration'], ascending=[False, True, True]).iloc[0]
    PRODUCTION_POLICY = {
        'threshold': float(production.threshold),
        'persistence': int(production.persistence),
        'maximum_wait': int(production.maximum_wait),
    }
else:
    PRODUCTION_POLICY = {
        'threshold': 0.5,
        'persistence': 1,
        'maximum_wait': online_handoff_config['bp_max_iterations'],
    }

PREFIX_TRIAL_PATH = online_handoff_trace_dir / 'prefix_buffer_trials.csv'
prefix_cache = online_handoff_stage_cached([PREFIX_TRIAL_PATH])
buffer_rows = []
prefix_groups = [] if prefix_cache else online_snapshot_features.groupby('shot_id', sort=False)

for online_handoff_shot_id, online_handoff_block in tqdm(prefix_groups, total=0 if prefix_cache else online_snapshot_features.shot_id.nunique(), desc='prefix buffers', disable=online_handoff_config['print_level'] == 1):
    online_handoff_block = online_handoff_block.sort_values('iteration')
    online_handoff_case = case_lookup.loc[str(online_handoff_shot_id)]
    online_handoff_endpoint = int(online_handoff_block.endpoint.iloc[0])
    prefix = online_handoff_block[online_handoff_block.iteration <= online_handoff_endpoint].copy()
    online_handoff_scores = score_block(prefix)
    online_handoff_context = online_handoff_context_for(int(online_handoff_case.level), float(online_handoff_case.p))
    online_handoff_syndrome = online_handoff_bit_array(online_handoff_case.syndrome, online_handoff_context.H.shape[0])
    online_handoff_trace = online_handoff_trace_min_sum(online_handoff_context, online_handoff_syndrome, online_handoff_config['bp_max_iterations'])

    for online_handoff_size in online_handoff_config['prefix_buffer_sizes']:
        online_handoff_candidates = prefix if online_handoff_size == -1 else prefix.tail(online_handoff_size)
        candidate_scores = online_handoff_scores[-len(online_handoff_candidates):]
        online_handoff_selected = int(online_handoff_candidates.iloc[int(np.argmax(candidate_scores))].iteration)
        online_handoff_correct, online_handoff_osd_runtime, online_handoff_ = evaluate_snapshot(online_handoff_case, online_handoff_selected, 1, online_handoff_trace)
        buffer_rows.append({
            'shot_id': online_handoff_shot_id,
            'distance': online_handoff_case.distance,
            'p': online_handoff_case.p,
            'split': online_handoff_case.split,
            'buffer_size': online_handoff_size,
            'decision_iteration': online_handoff_endpoint,
            'selected_iteration': online_handoff_selected,
            'logical_correct': online_handoff_correct,
            'bp_runtime_seconds': online_handoff_trace.runtime_seconds[online_handoff_endpoint],
            'osd_runtime_seconds': online_handoff_osd_runtime,
            'total_runtime_seconds': online_handoff_trace.runtime_seconds[online_handoff_endpoint] + online_handoff_osd_runtime,
            'candidate_count': online_handoff_candidate_count(online_handoff_context, 1),
        })

if prefix_cache:
    online_handoff_prefix_buffer_trials = online_handoff_read_csv(PREFIX_TRIAL_PATH)
else:
    online_handoff_prefix_buffer_trials = pd.DataFrame(buffer_rows)
    online_handoff_save_csv(online_handoff_prefix_buffer_trials, PREFIX_TRIAL_PATH)

prefix_buffer_selection = online_handoff_prefix_buffer_trials.groupby(['distance', 'p', 'split', 'buffer_size'], as_index=False).agg(
    shots=('shot_id', 'nunique'),
    success_rate=('logical_correct', 'mean'),
    mean_snapshot_age=('decision_iteration', lambda x: np.nan),
    mean_selected_iteration=('selected_iteration', 'mean'),
    mean_runtime_seconds=('total_runtime_seconds', 'mean'),
)
for online_handoff_index, online_handoff_row in prefix_buffer_selection.iterrows():
    subset = online_handoff_prefix_buffer_trials[(online_handoff_prefix_buffer_trials.distance == online_handoff_row.distance) & (online_handoff_prefix_buffer_trials.p == online_handoff_row.p) & (online_handoff_prefix_buffer_trials.split == online_handoff_row.split) & (online_handoff_prefix_buffer_trials.buffer_size == online_handoff_row.buffer_size)]
    prefix_buffer_selection.loc[online_handoff_index, 'mean_snapshot_age'] = (subset.decision_iteration - subset.selected_iteration).mean()

online_handoff_save_csv(prefix_buffer_selection, online_handoff_data_paths.output('prefix_buffer_selection_csv'))
ADAPTIVE_PATH = online_handoff_data_paths.output('adaptive_osd_order_csv')
adaptive_cache = online_handoff_stage_cached([ADAPTIVE_PATH])
adaptive_rows = []
adaptive_groups = [] if adaptive_cache else online_snapshot_features.groupby('shot_id', sort=False)

for online_handoff_shot_id, online_handoff_block in adaptive_groups:
    online_handoff_block = online_handoff_block.sort_values('iteration').reset_index(drop=True)
    online_handoff_case = case_lookup.loc[str(online_handoff_shot_id)]
    online_handoff_p1 = score_block(online_handoff_block, 'y_osd1')
    online_handoff_p0 = score_block(online_handoff_block, 'y_osd0')
    online_handoff_selected, reason = first_crossing(online_handoff_block, online_handoff_p1, **PRODUCTION_POLICY)
    position = int(np.flatnonzero(online_handoff_block.iteration.to_numpy() == online_handoff_selected)[0])
    online_handoff_context = online_handoff_context_for(int(online_handoff_case.level), float(online_handoff_case.p))
    online_handoff_syndrome = online_handoff_bit_array(online_handoff_case.syndrome, online_handoff_context.H.shape[0])
    online_handoff_trace = online_handoff_trace_min_sum(online_handoff_context, online_handoff_syndrome, online_handoff_config['bp_max_iterations'])

    for online_handoff_threshold in online_handoff_config['adaptive_order_thresholds']:
        online_handoff_order = online_handoff_full_order(online_handoff_context) if reason != 'trigger' else 0 if online_handoff_p0[position] >= online_handoff_threshold else 1 if online_handoff_p1[position] >= online_handoff_threshold else online_handoff_full_order(online_handoff_context)
        online_handoff_correct, online_handoff_osd_runtime, online_handoff_ = evaluate_snapshot(online_handoff_case, online_handoff_selected, online_handoff_order, online_handoff_trace)
        adaptive_rows.append({
            'shot_id': online_handoff_shot_id,
            'distance': online_handoff_case.distance,
            'p': online_handoff_case.p,
            'split': online_handoff_case.split,
            'threshold': online_handoff_threshold,
            'selected_iteration': online_handoff_selected,
            'osd_order': online_handoff_order,
            'logical_correct': online_handoff_correct,
            'candidate_count': online_handoff_candidate_count(online_handoff_context, online_handoff_order),
            'runtime_seconds': online_handoff_trace.runtime_seconds[online_handoff_selected] + online_handoff_osd_runtime,
        })

if adaptive_cache:
    adaptive_osd_order = online_handoff_read_csv(ADAPTIVE_PATH)
else:
    online_handoff_adaptive_trials = pd.DataFrame(adaptive_rows)
    adaptive_osd_order = online_handoff_adaptive_trials.groupby(['distance', 'p', 'split', 'threshold'], as_index=False).agg(
        shots=('shot_id', 'nunique'),
        success_rate=('logical_correct', 'mean'),
        mean_osd_order=('osd_order', 'mean'),
        fraction_osd0=('osd_order', lambda x: np.mean(x == 0)),
        fraction_osd1=('osd_order', lambda x: np.mean(x == 1)),
        fraction_full=('osd_order', lambda x: np.mean(x > 1)),
        mean_candidate_count=('candidate_count', 'mean'),
        mean_runtime_seconds=('runtime_seconds', 'mean'),
    )
    online_handoff_save_csv(adaptive_osd_order, ADAPTIVE_PATH)

instability_value_ablation = snapshot_feature_ablation[(snapshot_feature_ablation.analysis == 'feature_set') & snapshot_feature_ablation.target.eq('y_osd1')].copy()
instability_value_ablation['ablation_level'] = instability_value_ablation.feature_set.map({
    'static': 'snapshot_only',
    'rank': 'snapshot_plus_rank_dynamics',
    'message': 'snapshot_plus_message_dynamics',
    'full': 'full_dynamics',
})
online_handoff_save_csv(instability_value_ablation, online_handoff_data_paths.output('instability_value_ablation_csv'))

display(online_handoff_rules[online_handoff_rules.split == 'test'].head())


## Experiments 8–10: BP soft-generator scan, endpoint/final comparison and final paired benchmark

Parameter scans use a fixed `parallel` schedule; the final benchmark policy and threshold are determined exclusively from the mechanism development split.

Compare the soft-generator settings and audited endpoint/fixed-budget references before the final paired benchmark. Include invocation and regression costs when interpreting a handoff improvement.


In [ ]:
def row_trace(row, scaling=1.0, damping=0.0):
    context = online_handoff_context_for(int(row.level), float(row.p))
    syndrome = online_handoff_bit_array(row.syndrome, context.H.shape[0])
    truth = online_handoff_bit_array(row.true_logical, context.L.shape[0])
    trace = online_handoff_trace_min_sum(context, syndrome, online_handoff_config['bp_max_iterations'], scaling=scaling, damping=damping)

    return (context, syndrome, truth, trace)

online_handoff_scan_cases = pd.concat([mechanism_cohort[mechanism_cohort.cohort == 'G'].head(online_handoff_config['parameter_scan_case_limit']), final_cases.head(online_handoff_config['parameter_scan_unbiased_limit'])], ignore_index=True).drop_duplicates('shot_id')
online_handoff_parameter_scan_path = online_handoff_data_paths.output('bp_soft_generator_parameter_scan_csv')
online_handoff_parameter_scan_cache = online_handoff_stage_cached([online_handoff_parameter_scan_path])
scan_rows = []
scan_iterator = [] if online_handoff_parameter_scan_cache else online_handoff_scan_cases.itertuples(index=False)

for online_handoff_case in tqdm(scan_iterator, total=0 if online_handoff_parameter_scan_cache else len(online_handoff_scan_cases), desc='BP parameter scan', disable=online_handoff_config['print_level'] == 1):
    for online_handoff_scaling, online_handoff_damping in product(online_handoff_config['scaling_values'], online_handoff_config['damping_values']):
        online_handoff_context, online_handoff_syndrome, online_handoff_truth, online_handoff_trace = row_trace(online_handoff_case, online_handoff_scaling, online_handoff_damping)
        online_handoff_endpoint = online_handoff_endpoint_of(online_handoff_trace)
        online_handoff_successes = {0: [], 1: [], online_handoff_full_order(online_handoff_context): []}

        for online_handoff_t in range(1, len(online_handoff_trace.llr)):
            for online_handoff_q in online_handoff_successes:
                online_handoff_correction, online_handoff_ = osd_from_reliability(online_handoff_context, online_handoff_syndrome, online_handoff_trace.llr[online_handoff_t], online_handoff_q)
                online_handoff_successes[online_handoff_q].append(online_handoff_outcome(online_handoff_context, online_handoff_correction, online_handoff_syndrome, online_handoff_truth)[1])

        selected_features = pd.DataFrame([prefix_features(online_handoff_context, online_handoff_trace, online_handoff_syndrome, t) for t in range(1, len(online_handoff_trace.llr))])
        online_handoff_scores = np.full(len(selected_features), 0.5)
        online_handoff_model = MODELS.get(('y_osd1', 'full', 'logistic'))

        if online_handoff_model is not None:
            online_handoff_scores = online_handoff_model.predict_proba(selected_features[online_handoff_feature_sets['full']].fillna(0).to_numpy())[:, 1]
        pseudo = selected_features.copy()
        pseudo['endpoint'] = online_handoff_endpoint
        online_handoff_selected, reason = first_crossing(pseudo, online_handoff_scores, **PRODUCTION_POLICY)
        online_order = 1 if reason == 'trigger' else online_handoff_full_order(online_handoff_context)
        online_correction, online_handoff_ = osd_from_reliability(online_handoff_context, online_handoff_syndrome, online_handoff_trace.llr[online_handoff_selected], online_order)
        scan_rows.append({
            'shot_id': online_handoff_case.shot_id,
            'population': 'mechanism' if getattr(online_handoff_case, 'cohort', '') == 'G' else 'unbiased',
            'distance': online_handoff_context.distance,
            'p': online_handoff_case.p,
            'scaling': online_handoff_scaling,
            'damping': online_handoff_damping,
            'schedule': online_handoff_config['schedule'],
            'endpoint': online_handoff_endpoint,
            'first_valid': online_handoff_trace.first_valid if online_handoff_trace.first_valid is not None else -1,
            'osd0_prefix_opportunity': int(max(online_handoff_successes[0])),
            'osd1_prefix_opportunity': int(max(online_handoff_successes[1])),
            'full_prefix_opportunity': int(max(online_handoff_successes[online_handoff_full_order(online_handoff_context)])),
            'first_osd1_useful': next((i + 1 for i, x in enumerate(online_handoff_successes[1]) if x), -1),
            'online_selected_iteration': online_handoff_selected,
            'online_correct': online_handoff_outcome(online_handoff_context, online_correction, online_handoff_syndrome, online_handoff_truth)[1],
        })

if online_handoff_parameter_scan_cache:
    bp_soft_generator_parameter_scan = online_handoff_read_csv(online_handoff_parameter_scan_path)
else:
    scan_trials = pd.DataFrame(scan_rows)
    bp_soft_generator_parameter_scan = scan_trials.groupby(['population', 'distance', 'p', 'scaling', 'damping', 'schedule'], as_index=False).agg(
        shots=('shot_id', 'nunique'),
        endpoint_mean=('endpoint', 'mean'),
        osd0_prefix_opportunity=('osd0_prefix_opportunity', 'mean'),
        osd1_prefix_opportunity=('osd1_prefix_opportunity', 'mean'),
        full_prefix_opportunity=('full_prefix_opportunity', 'mean'),
        online_success_rate=('online_correct', 'mean'),
        mean_online_iteration=('online_selected_iteration', 'mean'),
    )
    online_handoff_save_csv(bp_soft_generator_parameter_scan, online_handoff_parameter_scan_path)

endpoint_rows = []
for online_handoff_case in final_cases.itertuples(index=False):
    for online_handoff_name in ['osd0', 'osd1', 'full']:
        endpoint_rows.append({
            'shot_id': online_handoff_case.shot_id,
            'distance': online_handoff_case.distance,
            'p': online_handoff_case.p,
            'osd_order': online_handoff_name,
            'endpoint_correct': int(getattr(online_handoff_case, f'forced_endpoint_{online_handoff_name}_correct')),
            'final_correct': int(getattr(online_handoff_case, f'final_Tmax_{online_handoff_name}_correct')),
        })

endpoint_trials = pd.DataFrame(endpoint_rows)
endpoint_vs_final_osd = endpoint_trials.groupby(['distance', 'p', 'osd_order'], as_index=False).agg(
    shots=('shot_id', 'nunique'),
    endpoint_success=('endpoint_correct', 'mean'),
    final_success=('final_correct', 'mean'),
    endpoint_fail_final_repair=('endpoint_correct', lambda x: np.nan),
    endpoint_success_final_regression=('endpoint_correct', lambda x: np.nan),
)
for online_handoff_index, online_handoff_row in endpoint_vs_final_osd.iterrows():
    online_handoff_block = endpoint_trials[(endpoint_trials.distance == online_handoff_row.distance) & (endpoint_trials.p == online_handoff_row.p) & (endpoint_trials.osd_order == online_handoff_row.osd_order)]
    endpoint_vs_final_osd.loc[online_handoff_index, 'endpoint_fail_final_repair'] = np.mean((online_handoff_block.endpoint_correct == 0) & (online_handoff_block.final_correct == 1))
    endpoint_vs_final_osd.loc[online_handoff_index, 'endpoint_success_final_regression'] = np.mean((online_handoff_block.endpoint_correct == 1) & (online_handoff_block.final_correct == 0))

online_handoff_save_csv(endpoint_vs_final_osd, online_handoff_data_paths.output('endpoint_vs_final_osd_csv'))

def evaluate_production_methods(case):
    context, syndrome, truth, trace = row_trace(case)
    endpoint = online_handoff_endpoint_of(trace)
    full = online_handoff_full_order(context)
    frame = pd.DataFrame([{
        **{'iteration': t, 'endpoint': endpoint},
        **prefix_features(context, trace, syndrome, t),
    } for t in range(1, len(trace.llr))])
    scores1 = np.full(len(frame), 0.5)
    scores0 = np.full(len(frame), 0.5)

    if MODELS.get(('y_osd1', 'full', 'logistic')) is not None:
        scores1 = MODELS['y_osd1', 'full', 'logistic'].predict_proba(frame[online_handoff_feature_sets['full']].fillna(0).to_numpy())[:, 1]
    if MODELS.get(('y_osd0', 'full', 'logistic')) is not None:
        scores0 = MODELS['y_osd0', 'full', 'logistic'].predict_proba(frame[online_handoff_feature_sets['full']].fillna(0).to_numpy())[:, 1]
    selected, reason = first_crossing(frame, scores1, **PRODUCTION_POLICY)
    matches = np.flatnonzero(frame.iteration.to_numpy() == selected)
    position = int(matches[0]) if len(matches) else 0
    dev_adaptive = adaptive_osd_order[adaptive_osd_order.split == 'development']
    adaptive_threshold = float(dev_adaptive.sort_values(['success_rate', 'mean_candidate_count'], ascending=[False, True]).iloc[0].threshold) if len(dev_adaptive) else 0.5
    adaptive_order = full if reason != 'trigger' else 0 if scores0[position] >= adaptive_threshold else 1 if scores1[position] >= adaptive_threshold else full
    dev_buffers = prefix_buffer_selection[prefix_buffer_selection.split == 'development']
    buffer_size = int(dev_buffers.sort_values(['success_rate', 'mean_selected_iteration'], ascending=[False, True]).iloc[0].buffer_size) if len(dev_buffers) else -1
    prefix = frame[frame.iteration <= endpoint]
    candidates = prefix if buffer_size == -1 else prefix.tail(buffer_size)

    if len(candidates):
        prefix_scores = scores1[:len(prefix)][-len(candidates):]
        buffer_iteration = int(candidates.iloc[int(np.argmax(prefix_scores))].iteration)
    else:
        buffer_iteration = 0

    methods = {
        'standard_BPOSD': (int(case.standard_correct), endpoint, full if case.first_valid < 0 else -1, float(case.bp_endpoint_runtime_seconds + case.standard_osd_runtime_seconds)),
        'forced_endpoint_OSD0': (int(case.forced_endpoint_osd0_correct), endpoint, 0, float(case.bp_endpoint_runtime_seconds + case.forced_endpoint_osd0_runtime_seconds)),
        'forced_endpoint_OSD1': (int(case.forced_endpoint_osd1_correct), endpoint, 1, float(case.bp_endpoint_runtime_seconds + case.forced_endpoint_osd1_runtime_seconds)),
        'forced_endpoint_OSDfull': (int(case.forced_endpoint_full_correct), endpoint, full, float(case.bp_endpoint_runtime_seconds + case.forced_endpoint_full_runtime_seconds)),
        'final_Tmax_OSD0': (int(case.final_Tmax_osd0_correct), online_handoff_config['bp_max_iterations'], 0, float(case.bp_final_runtime_seconds + case.final_Tmax_osd0_runtime_seconds)),
        'final_Tmax_OSD1': (int(case.final_Tmax_osd1_correct), online_handoff_config['bp_max_iterations'], 1, float(case.bp_final_runtime_seconds + case.final_Tmax_osd1_runtime_seconds)),
        'final_Tmax_OSDfull': (int(case.final_Tmax_full_correct), online_handoff_config['bp_max_iterations'], full, float(case.bp_final_runtime_seconds + case.final_Tmax_full_runtime_seconds)),
        'Tesseract': (int(case.tesseract_correct), -1, -1, float(case.tesseract_runtime_seconds)),
    }

    for name, iteration, order in [('online_handoff_OSD1', selected, 1), ('prefix_buffer_OSD1', buffer_iteration, 1), ('adaptive_online_OSD', selected, adaptive_order)]:
        correction, osd_runtime = osd_from_reliability(context, syndrome, trace.llr[iteration], order)
        methods[name] = (online_handoff_outcome(context, correction, syndrome, truth)[1], iteration, order, float(trace.runtime_seconds[iteration] + osd_runtime))

    return methods

FINAL_BENCHMARK_PATH = online_handoff_data_paths.output('final_gap_benchmark_csv')
ACCURACY_COST_PATH = online_handoff_data_paths.output('accuracy_cost_frontier_csv')
final_benchmark_cache = online_handoff_stage_cached([FINAL_BENCHMARK_PATH, ACCURACY_COST_PATH])
online_handoff_benchmark_rows = []
benchmark_iterator = [] if final_benchmark_cache else final_cases.itertuples(index=False)

for online_handoff_case in tqdm(benchmark_iterator, total=0 if final_benchmark_cache else len(final_cases), desc='final paired benchmark', disable=online_handoff_config['print_level'] == 1):
    for online_handoff_decoder, (online_handoff_correct, iteration, online_handoff_order, online_handoff_runtime) in evaluate_production_methods(online_handoff_case).items():
        online_handoff_context = online_handoff_context_for(int(online_handoff_case.level), float(online_handoff_case.p))
        online_handoff_benchmark_rows.append({
            'shot_id': online_handoff_case.shot_id,
            'distance': online_handoff_case.distance,
            'p': online_handoff_case.p,
            'decoder': online_handoff_decoder,
            'logical_correct': online_handoff_correct,
            'iteration': iteration,
            'osd_order': online_handoff_order,
            'candidate_count': 0 if online_handoff_order < 0 else online_handoff_candidate_count(online_handoff_context, online_handoff_order),
            'runtime_seconds': online_handoff_runtime,
            'standard_correct': int(online_handoff_case.standard_correct),
            'tesseract_correct': int(online_handoff_case.tesseract_correct),
        })

if final_benchmark_cache:
    final_gap_benchmark = online_handoff_read_csv(FINAL_BENCHMARK_PATH)
    accuracy_cost_frontier = online_handoff_read_csv(ACCURACY_COST_PATH)
else:
    benchmark_trials = pd.DataFrame(online_handoff_benchmark_rows)
    online_handoff_final_rows = []
    for (online_handoff_distance, online_handoff_p), online_handoff_point in benchmark_trials.groupby(['distance', 'p']):
        online_handoff_standard = online_handoff_point[online_handoff_point.decoder == 'standard_BPOSD'].set_index('shot_id').logical_correct
        online_handoff_tess = online_handoff_point[online_handoff_point.decoder == 'Tesseract'].set_index('shot_id').logical_correct
        online_handoff_gap, online_handoff_lower, online_handoff_upper = online_handoff_paired_gap_interval(online_handoff_standard, online_handoff_tess)
        online_handoff_gap_status = 'resolved' if online_handoff_lower > 0 else 'unresolved'

        for online_handoff_decoder, online_handoff_block in online_handoff_point.groupby('decoder'):
            indexed = online_handoff_block.set_index('shot_id')
            aligned_standard = online_handoff_standard.loc[indexed.index]
            aligned_tess = online_handoff_tess.loc[indexed.index]
            wrong_gap = (aligned_standard == 0) & (aligned_tess == 1)
            standard_success = aligned_standard == 1
            repair = np.mean(indexed.logical_correct[wrong_gap] == 1) if wrong_gap.any() else np.nan
            regression = np.mean(indexed.logical_correct[standard_success] == 0) if standard_success.any() else np.nan
            online_handoff_ler = 1 - indexed.logical_correct.mean()
            closure = (1 - online_handoff_standard.mean() - online_handoff_ler) / online_handoff_gap if online_handoff_gap_status == 'resolved' else np.nan
            online_handoff_final_rows.append({
                'distance': online_handoff_distance,
                'p': online_handoff_p,
                'decoder': online_handoff_decoder,
                'shots': len(online_handoff_block),
                'LER': online_handoff_ler,
                'standard_LER': 1 - online_handoff_standard.mean(),
                'Tesseract_LER': 1 - online_handoff_tess.mean(),
                'paired_gap': online_handoff_gap,
                'paired_gap_ci_low': online_handoff_lower,
                'paired_gap_ci_high': online_handoff_upper,
                'gap_status': online_handoff_gap_status,
                'gap_closure_fraction': closure,
                'conditional_repair_rate': repair,
                'conditional_regression_rate': regression,
                'mean_iteration': online_handoff_block.iteration.replace(-1, np.nan).mean(),
                'mean_osd_order': online_handoff_block.osd_order.replace(-1, np.nan).mean(),
                'fraction_full_osd': np.mean(online_handoff_block.osd_order > 1),
                'mean_candidate_count': online_handoff_block.candidate_count.mean(),
                'mean_runtime_seconds': online_handoff_block.runtime_seconds.mean(),
                'median_runtime_seconds': online_handoff_block.runtime_seconds.median(),
                'p95_runtime_seconds': online_handoff_block.runtime_seconds.quantile(0.95),
            })

    final_gap_benchmark = pd.DataFrame(online_handoff_final_rows)
    online_handoff_save_csv(final_gap_benchmark, FINAL_BENCHMARK_PATH)
    accuracy_cost_frontier = pd.concat(
        [final_gap_benchmark.assign(source='final_decoder')[['source', 'distance', 'p', 'decoder', 'LER', 'gap_closure_fraction', 'fraction_full_osd', 'mean_osd_order', 'mean_candidate_count', 'mean_runtime_seconds']], adaptive_osd_order[adaptive_osd_order.split == 'test'].assign(
            source='adaptive_threshold',
            decoder=lambda x: 'adaptive_tau=' + x.threshold.astype(str),
            LER=lambda x: 1 - x.success_rate,
            gap_closure_fraction=np.nan,
        )[['source', 'distance', 'p', 'decoder', 'LER', 'gap_closure_fraction', 'fraction_full', 'mean_osd_order', 'mean_candidate_count', 'mean_runtime_seconds']].rename(columns={'fraction_full': 'fraction_full_osd'})],
        ignore_index=True,
    )
    online_handoff_save_csv(accuracy_cost_frontier, ACCURACY_COST_PATH)

display(final_gap_benchmark)


## Required figures and completeness checks

Figures are displayed inline only and are not saved to disk. If the formal d=9 strict-gap count is below the threshold, its results must be labeled exploratory and cannot support a distance-generalization claim.

In [ ]:
online_handoff_required_stats = ['qmin_temporal_summary.csv', 'prefix_oracle_ceiling.csv', 'snapshot_feature_ablation.csv', 'temporal_shuffle_control.csv', 'online_handoff_rules.csv', 'prefix_buffer_selection.csv', 'adaptive_osd_order.csv', 'instability_value_ablation.csv', 'bp_soft_generator_parameter_scan.csv', 'endpoint_vs_final_osd.csv', 'final_gap_benchmark.csv', 'accuracy_cost_frontier.csv']
online_handoff_required_trace = ['qmin_snapshots.csv', 'online_snapshot_features.csv', 'online_handoff_trials.csv', 'prefix_buffer_trials.csv']
assert all((online_handoff_data_paths.output('name', name=name).is_file() for name in online_handoff_required_stats))
assert all(((online_handoff_trace_dir / name).is_file() for name in online_handoff_required_trace))

online_handoff_fig, online_handoff_axes = plt.subplots(2, 3, figsize=(17, 9))
gap_q = qmin_snapshots[qmin_snapshots.cohort == 'G'].copy()
if len(gap_q):
    pivot = gap_q.assign(q_value=gap_q.q_min.map({
        '0': 0,
        '1': 1,
        'full': 2,
        'inf': 3,
    })).pivot(index='shot_id', columns='iteration', values='q_value')

    sns.heatmap(pivot.head(100), cmap='viridis', cbar_kws={'ticks': [0, 1, 2, 3]}, ax=online_handoff_axes[0, 0])
    online_handoff_axes[0, 0].set_title('Figure 1: q_min trajectories (strict gap)')
    aligned = gap_q.groupby('relative_to_first_valid')[['osd0_correct', 'osd1_correct']].mean().reset_index()

    online_handoff_axes[0, 1].plot(aligned.relative_to_first_valid, aligned.osd0_correct, label='OSD0')
    online_handoff_axes[0, 1].plot(aligned.relative_to_first_valid, aligned.osd1_correct, label='OSD1')
    online_handoff_axes[0, 1].legend()
    online_handoff_axes[0, 1].set_title('Figure 2: first-valid aligned success')
else:
    online_handoff_axes[0, 0].text(0.5, 0.5, 'No strict-gap events', ha='center')
    online_handoff_axes[0, 1].text(0.5, 0.5, 'No strict-gap events', ha='center')

test_ablation = snapshot_feature_ablation[(snapshot_feature_ablation.analysis == 'feature_set') & (snapshot_feature_ablation.target == 'y_osd1') & (snapshot_feature_ablation.model == 'logistic')]
sns.barplot(test_ablation, x='feature_set', y='test_auprc', ax=online_handoff_axes[0, 2])
online_handoff_axes[0, 2].set_title('Figure 3: snapshot-model ablation')
test_rules = online_handoff_rules[(online_handoff_rules.split == 'test') & (online_handoff_rules.policy_family == 'learned_full_logistic') & (online_handoff_rules.require_syndrome_consistent == 0)]

if len(test_rules):
    sns.scatterplot(
        test_rules,
        x='mean_selected_iteration',
        y='success_rate',
        hue='threshold',
        size='persistence',
        ax=online_handoff_axes[1, 0],
    )
online_handoff_axes[1, 0].set_title('Figure 4: online handoff frontier')

if len(prefix_buffer_selection):
    sns.lineplot(
        prefix_buffer_selection[prefix_buffer_selection.split == 'test'],
        x='buffer_size',
        y='success_rate',
        hue='p',
        marker='o',
        ax=online_handoff_axes[1, 1],
    )
online_handoff_axes[1, 1].set_title('Figure 5: causal prefix buffer')
frontier = accuracy_cost_frontier[accuracy_cost_frontier.source == 'final_decoder']

if len(frontier):
    sns.scatterplot(
        frontier,
        x='mean_runtime_seconds',
        y='LER',
        hue='decoder',
        style='distance',
        ax=online_handoff_axes[1, 2],
    )
online_handoff_axes[1, 2].set_title('Figure 6: final accuracy–cost frontier')
plt.tight_layout()

plt.show()
d9_gap_count = int(((mechanism_population.distance == 9) & (mechanism_population.cohort == 'G')).sum())
d9_status = 'confirmatory' if d9_gap_count >= online_handoff_config['minimum_d9_strict_gap_events'] else 'exploratory'
online_handoff_log(2, {
    'status': 'complete',
    'd5_primary': True,
    'd9_strict_gap_events': d9_gap_count,
    'd9_status': d9_status,
    'production_policy': PRODUCTION_POLICY,
    'full_osd': {
        'method': online_handoff_config['full_osd_method'],
        'orders': online_handoff_config['full_osd_order_by_distance'],
        'combination_sweep': online_handoff_config['full_osd_combination_sweep_setting'],
        'candidate_budgets': online_handoff_config['full_osd_candidate_budget_by_distance'],
    },
})
